# Explainability Evaluation - Reconstruction Error Agreement

## Testing the Attention based Explainability and SHAP on Agreement with Reconstruction Error

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Connect and Access Git Repository

import os
import sys
import json
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"

token = getpass("GitHub token: ")

REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
Cloning into '/content/bachelor-thesis'...
remote: Enumerating objects: 751, done.
remote: Counting objects: 100% (266/266), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 751 (delta 172), reused 128 (delta 70), pack-reused 485 (from 2)
Receiving objects: 100% (751/751), 37.75 MiB | 17.28 MiB/s, done.
Resolving deltas: 100% (412/412), done.
/content/bachelor-thesis
Repo exists: True


In [3]:
import os
import numpy as np
import pandas as pd
import shap
import torch
from torch.utils.data import DataLoader, TensorDataset

# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))
sys.path.append(str(REPO_DIR / "explainability"))


from utils.dataset import PTBXLDataset
from utils.seed import set_seed, RANDOM_SEED
from explainability.anoformer_attention import AnoFormerExplainability
from explainability.tranad_attention import TranADExplainability
from models.anoformer import AnoFormerMainModel
from models.tranad import TranADMainModel
from explainability.evaluation import ReconstructionErrorAgreementEvaluation
from explainability.sample_selection import load_samples_from_dataset


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [5]:
# Paths to Directories
# Load processed data and pretrained models

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"
SHAP_RESULTS_DIR = RESULTS_DIR / "shap"

MODEL_DIR = RESULTS_DIR / "models"
ANOFORMER_MODEL_PATH = MODEL_DIR / "anoformer" / "anoformer_40epochs_larger_capacity_2.pt"
TRANAD_MODEL_PATH = MODEL_DIR / "tranad" / "tranad_20epochs_larger_capacity.pt"

AGREEMENT_DIR = RESULTS_DIR / "evaluation" / "reconstruction_error_agreement"
AGREEMENT_DIR .mkdir(parents=True, exist_ok=True)

In [6]:
# Load my preprocessed training and test datasets

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2198, 1000, 12) (2198,)


In [7]:
# select only normal ecgs from Training set
normal_train_mask = y_train == 0
X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

# Datasets
# We need the Train dataset because the SHAP background is from the training set. We also need the mean and std of the training set to normalize the test set.
train_dataset = PTBXLDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)

# We need the Test dataset because the chosen samples for evaluation are from the test set
test_dataset = PTBXLDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [8]:
# Load fixed evaluation samples

EvaluationSamplePath = (
    RESULTS_DIR
    / "evaluation"
    / "evaluation_samples.npz"
)

EvaluationSamples = load_samples_from_dataset(
    sample_path=EvaluationSamplePath,
    test_dataset=test_dataset,
    device=None,
)

print("Number of evaluation samples:", len(EvaluationSamples))

for sample in EvaluationSamples:
    print(
        sample["index"],
        sample["case_type"],
        sample["label"],
        sample["ecg"].shape,
    )

Number of evaluation samples: 20
2154 TN 0 torch.Size([1, 1000, 12])
1647 TN 0 torch.Size([1, 1000, 12])
161 TN 0 torch.Size([1, 1000, 12])
1394 TN 0 torch.Size([1, 1000, 12])
929 TN 0 torch.Size([1, 1000, 12])
911 TN 0 torch.Size([1, 1000, 12])
1528 TN 0 torch.Size([1, 1000, 12])
193 TN 0 torch.Size([1, 1000, 12])
383 TN 0 torch.Size([1, 1000, 12])
1878 TN 0 torch.Size([1, 1000, 12])
1134 TP 1 torch.Size([1, 1000, 12])
1854 TP 1 torch.Size([1, 1000, 12])
102 TP 1 torch.Size([1, 1000, 12])
1310 TP 1 torch.Size([1, 1000, 12])
1462 TP 1 torch.Size([1, 1000, 12])
1041 TP 1 torch.Size([1, 1000, 12])
56 TP 1 torch.Size([1, 1000, 12])
653 TP 1 torch.Size([1, 1000, 12])
910 TP 1 torch.Size([1, 1000, 12])
1332 TP 1 torch.Size([1, 1000, 12])


In [9]:
# Create the Evaluator instance

AgreementEvaluator = ReconstructionErrorAgreementEvaluation()

# AnoFormer

In [10]:
# Create AnoFormer Model

AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

# Load the pretrained model
AnoFormer.Load(str(ANOFORMER_MODEL_PATH))

# Create Explainer to extract Attention Values
AnoFormerExplainer = AnoFormerExplainability(AnoFormer)


Model loaded from: /content/drive/MyDrive/ECG data/results/models/anoformer/anoformer_40epochs_larger_capacity_2.pt


In [11]:
# Extract the reconstruction Error for the normal and anomalous ECGs
AnoFormerReconstructionErrorValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(AnoFormer.Device)

    _, _, reconstruction_error = (
        AnoFormerExplainer.get_reconstruction_error_over_time(
            TimeSeriesWindow=ecg,
            DatasetForInverseNorm=test_dataset,
        )
    )

    AnoFormerReconstructionErrorValues.append(
        reconstruction_error
    )

    print(
        "AnoFormer reconstruction error:",
        sample["index"],
        sample["case_type"],
        reconstruction_error.shape,
    )

AnoFormer reconstruction error: 2154 TN (1000,)
AnoFormer reconstruction error: 1647 TN (1000,)
AnoFormer reconstruction error: 161 TN (1000,)
AnoFormer reconstruction error: 1394 TN (1000,)
AnoFormer reconstruction error: 929 TN (1000,)
AnoFormer reconstruction error: 911 TN (1000,)
AnoFormer reconstruction error: 1528 TN (1000,)
AnoFormer reconstruction error: 193 TN (1000,)
AnoFormer reconstruction error: 383 TN (1000,)
AnoFormer reconstruction error: 1878 TN (1000,)
AnoFormer reconstruction error: 1134 TP (1000,)
AnoFormer reconstruction error: 1854 TP (1000,)
AnoFormer reconstruction error: 102 TP (1000,)
AnoFormer reconstruction error: 1310 TP (1000,)
AnoFormer reconstruction error: 1462 TP (1000,)
AnoFormer reconstruction error: 1041 TP (1000,)
AnoFormer reconstruction error: 56 TP (1000,)
AnoFormer reconstruction error: 653 TP (1000,)
AnoFormer reconstruction error: 910 TP (1000,)
AnoFormer reconstruction error: 1332 TP (1000,)


### Attention Evaluation

In [12]:
# Extract the attention importance values for the normal and anomalous ECGs

AnoFormerAttentionImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(AnoFormer.Device)

    attention_importance = AnoFormerExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        BlockIndex=0,
        HeadIndex=None,
    )

    AnoFormerAttentionImportanceValues.append(
        attention_importance
    )

    print(
        "AnoFormer attention:",
        sample["index"],
        sample["case_type"],
        attention_importance.shape,
    )

AnoFormer attention: 2154 TN (1000,)
AnoFormer attention: 1647 TN (1000,)
AnoFormer attention: 161 TN (1000,)
AnoFormer attention: 1394 TN (1000,)
AnoFormer attention: 929 TN (1000,)
AnoFormer attention: 911 TN (1000,)
AnoFormer attention: 1528 TN (1000,)
AnoFormer attention: 193 TN (1000,)
AnoFormer attention: 383 TN (1000,)
AnoFormer attention: 1878 TN (1000,)
AnoFormer attention: 1134 TP (1000,)
AnoFormer attention: 1854 TP (1000,)
AnoFormer attention: 102 TP (1000,)
AnoFormer attention: 1310 TP (1000,)
AnoFormer attention: 1462 TP (1000,)
AnoFormer attention: 1041 TP (1000,)
AnoFormer attention: 56 TP (1000,)
AnoFormer attention: 653 TP (1000,)
AnoFormer attention: 910 TP (1000,)
AnoFormer attention: 1332 TP (1000,)


In [13]:
# Evaluate

AnoFormerAttentionAgreementList = (
    AgreementEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=AnoFormerAttentionImportanceValues,
        reconstruction_error_values=AnoFormerReconstructionErrorValues,
        model_name="AnoFormer",
        explanation_name="Attention",
    )
)

AnoFormerAttentionAgreementResults = pd.DataFrame(
    AnoFormerAttentionAgreementList
)

AnoFormerAttentionAgreementResults

,Model,Explanation,Index,Case Type,Label,Spearman Correlation
0,AnoFormer,Attention,2154,TN,0,0.324792
1,AnoFormer,Attention,1647,TN,0,0.171969
2,AnoFormer,Attention,161,TN,0,0.256334
3,AnoFormer,Attention,1394,TN,0,0.422072
4,AnoFormer,Attention,929,TN,0,0.210475
5,AnoFormer,Attention,911,TN,0,0.350641
6,AnoFormer,Attention,1528,TN,0,0.225618
7,AnoFormer,Attention,193,TN,0,0.144962
8,AnoFormer,Attention,383,TN,0,0.423979
9,AnoFormer,Attention,1878,TN,0,0.270647


In [14]:
AnoFormerAttentionAgreementSummary = (
    AnoFormerAttentionAgreementResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Spearman Correlation"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerAttentionAgreementSummary

,Model,Explanation,Case Type,mean,std,count
0,AnoFormer,Attention,TN,0.280149,0.097898,10
1,AnoFormer,Attention,TP,0.230699,0.091359,10


In [15]:
AnoFormerAttentionAgreementResults.to_csv(
    AGREEMENT_DIR / "anoformer_attention_reconstruction_error_agreement_results.csv",
    index=False,
)

AnoFormerAttentionAgreementSummary.to_csv(
    AGREEMENT_DIR / "anoformer_attention_reconstruction_error_agreement_summary.csv",
    index=False,
)

## SHAP Evaluation

In [16]:
# Load shap values

AnoFormerSHAPValues = np.load(
    SHAP_RESULTS_DIR
    / "anoformer"
    / "anoformer_evaluation_shap_values.npy"
)

print("AnoFormer SHAP values:", AnoFormerSHAPValues.shape)

AnoFormer SHAP values: (20, 1000, 12)


In [17]:
# transform SHAP values temporal importance

AnoFormerSHAPImportanceValues = []

for shap_values in AnoFormerSHAPValues:
    temporal_importance = np.abs(
        shap_values
    ).sum(axis=1)

    AnoFormerSHAPImportanceValues.append(
        temporal_importance
    )

print(
    "Number of AnoFormer SHAP importance arrays:",
    len(AnoFormerSHAPImportanceValues),
)

print(
    "Shape of first AnoFormer SHAP importance:",
    AnoFormerSHAPImportanceValues[0].shape,
)

Number of AnoFormer SHAP importance arrays: 20
Shape of first AnoFormer SHAP importance: (1000,)


In [18]:
# Evaluate

AnoFormerSHAPAgreementList = (
    AgreementEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=AnoFormerSHAPImportanceValues,
        reconstruction_error_values=AnoFormerReconstructionErrorValues,
        model_name="AnoFormer",
        explanation_name="SHAP",
    )
)

AnoFormerSHAPAgreementResults = pd.DataFrame(
    AnoFormerSHAPAgreementList
)

AnoFormerSHAPAgreementResults

,Model,Explanation,Index,Case Type,Label,Spearman Correlation
0,AnoFormer,SHAP,2154,TN,0,0.328219
1,AnoFormer,SHAP,1647,TN,0,0.251029
2,AnoFormer,SHAP,161,TN,0,0.280455
3,AnoFormer,SHAP,1394,TN,0,0.219525
4,AnoFormer,SHAP,929,TN,0,0.427460
5,AnoFormer,SHAP,911,TN,0,0.303592
6,AnoFormer,SHAP,1528,TN,0,0.291437
7,AnoFormer,SHAP,193,TN,0,0.252731
8,AnoFormer,SHAP,383,TN,0,0.490318
9,AnoFormer,SHAP,1878,TN,0,0.319591


In [19]:
AnoFormerSHAPAgreementSummary = (
    AnoFormerSHAPAgreementResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Spearman Correlation"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerSHAPAgreementSummary

,Model,Explanation,Case Type,mean,std,count
0,AnoFormer,SHAP,TN,0.316436,0.083348,10
1,AnoFormer,SHAP,TP,0.465382,0.117179,10


In [20]:
# Save the Results

AnoFormerSHAPAgreementResults.to_csv(
    AGREEMENT_DIR / "anoformer_shap_reconstruction_error_agreement_results.csv",
    index=False,
)

AnoFormerSHAPAgreementSummary.to_csv(
    AGREEMENT_DIR / "anoformer_shap_reconstruction_error_agreement_summary.csv",
    index=False,
)

# TranAD

In [21]:
# Create TranAD Model

TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

# Load the pretrained model
TranAD.Load(str(TRANAD_MODEL_PATH))

# Create Explainer to extract Attention Values
TranADExplainer = TranADExplainability(TranAD)

TranADModel: 268,376 Parameter | d_model=64, H=4, Enc=2, Dec×2=2
Model loaded  /content/drive/MyDrive/ECG data/results/models/tranad/tranad_20epochs_larger_capacity.pt


In [22]:
# Extract the reconstruction Error for the normal and anomalous ECGs

TranADReconstructionErrorValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(TranAD.Device)

    _, _, reconstruction_error = (
        TranADExplainer.get_reconstruction_error_over_time(
            TimeSeriesWindow=ecg,
            DatasetForInverseNorm=test_dataset,
        )
    )

    TranADReconstructionErrorValues.append(
        reconstruction_error
    )

    print(
        "TranAD reconstruction error:",
        sample["index"],
        sample["case_type"],
        reconstruction_error.shape,
    )


TranAD reconstruction error: 2154 TN (1000,)
TranAD reconstruction error: 1647 TN (1000,)
TranAD reconstruction error: 161 TN (1000,)
TranAD reconstruction error: 1394 TN (1000,)
TranAD reconstruction error: 929 TN (1000,)
TranAD reconstruction error: 911 TN (1000,)
TranAD reconstruction error: 1528 TN (1000,)
TranAD reconstruction error: 193 TN (1000,)
TranAD reconstruction error: 383 TN (1000,)
TranAD reconstruction error: 1878 TN (1000,)
TranAD reconstruction error: 1134 TP (1000,)
TranAD reconstruction error: 1854 TP (1000,)
TranAD reconstruction error: 102 TP (1000,)
TranAD reconstruction error: 1310 TP (1000,)
TranAD reconstruction error: 1462 TP (1000,)
TranAD reconstruction error: 1041 TP (1000,)
TranAD reconstruction error: 56 TP (1000,)
TranAD reconstruction error: 653 TP (1000,)
TranAD reconstruction error: 910 TP (1000,)
TranAD reconstruction error: 1332 TP (1000,)


### Attention Evaluation

In [23]:
# Extract Attention Values of Original ECGs

TranADAttentionImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(TranAD.Device)

    attention_importance = TranADExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        LayerIndex=0,
        HeadIndex=None,
    )

    TranADAttentionImportanceValues.append(
        attention_importance
    )

    print(
        "TranAD attention:",
        sample["index"],
        sample["case_type"],
        attention_importance.shape,
    )

TranAD attention: 2154 TN (1000,)
TranAD attention: 1647 TN (1000,)
TranAD attention: 161 TN (1000,)
TranAD attention: 1394 TN (1000,)
TranAD attention: 929 TN (1000,)
TranAD attention: 911 TN (1000,)
TranAD attention: 1528 TN (1000,)
TranAD attention: 193 TN (1000,)
TranAD attention: 383 TN (1000,)
TranAD attention: 1878 TN (1000,)
TranAD attention: 1134 TP (1000,)
TranAD attention: 1854 TP (1000,)
TranAD attention: 102 TP (1000,)
TranAD attention: 1310 TP (1000,)
TranAD attention: 1462 TP (1000,)
TranAD attention: 1041 TP (1000,)
TranAD attention: 56 TP (1000,)
TranAD attention: 653 TP (1000,)
TranAD attention: 910 TP (1000,)
TranAD attention: 1332 TP (1000,)


In [24]:
# Evaluate

TranADAttentionAgreementList = (
    AgreementEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADAttentionImportanceValues,
        reconstruction_error_values=TranADReconstructionErrorValues,
        model_name="TranAD",
        explanation_name="Attention",
    )
)

TranADAttentionAgreementResults = pd.DataFrame(
    TranADAttentionAgreementList
)

TranADAttentionAgreementResults

,Model,Explanation,Index,Case Type,Label,Spearman Correlation
0,TranAD,Attention,2154,TN,0,0.543101
1,TranAD,Attention,1647,TN,0,0.549452
2,TranAD,Attention,161,TN,0,0.404810
3,TranAD,Attention,1394,TN,0,0.642978
4,TranAD,Attention,929,TN,0,0.171740
5,TranAD,Attention,911,TN,0,0.372992
6,TranAD,Attention,1528,TN,0,0.418332
7,TranAD,Attention,193,TN,0,0.568807
8,TranAD,Attention,383,TN,0,0.494007
9,TranAD,Attention,1878,TN,0,0.608962


In [25]:
TranADAttentionAgreementSummary = (
    TranADAttentionAgreementResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Spearman Correlation"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADAttentionAgreementSummary

,Model,Explanation,Case Type,mean,std,count
0,TranAD,Attention,TN,0.477518,0.139822,10
1,TranAD,Attention,TP,0.583823,0.156952,10


In [26]:
# Save the Results

TranADAttentionAgreementResults.to_csv(
    AGREEMENT_DIR / "tranad_attention_reconstruction_error_agreement_results.csv",
    index=False,
)

TranADAttentionAgreementSummary.to_csv(
    AGREEMENT_DIR / "tranad_attention_reconstruction_error_agreement_summary.csv",
    index=False,
)

## SHAP Evaluation

In [27]:
# Load shap values

TranADSHAPValues = np.load(
    SHAP_RESULTS_DIR
    / "tranad"
    / "tranad_evaluation_shap_values.npy"
)

print("TranAD SHAP values:", TranADSHAPValues.shape)

TranAD SHAP values: (20, 1000, 12)


In [28]:
# transform SHAP values into temporal importance

TranADSHAPImportanceValues = []

for shap_values in TranADSHAPValues:
    temporal_importance = np.abs(
        shap_values
    ).sum(axis=1)

    TranADSHAPImportanceValues.append(
        temporal_importance
    )

print(
    "Number of TranAD SHAP importance arrays:",
    len(TranADSHAPImportanceValues),
)

print(
    "Shape of first TranAD SHAP importance:",
    TranADSHAPImportanceValues[0].shape,
)

Number of TranAD SHAP importance arrays: 20
Shape of first TranAD SHAP importance: (1000,)


In [29]:
# Evaluate

TranADSHAPAgreementList = (
    AgreementEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADSHAPImportanceValues,
        reconstruction_error_values=TranADReconstructionErrorValues,
        model_name="TranAD",
        explanation_name="SHAP",
    )
)

TranADSHAPAgreementResults = pd.DataFrame(
    TranADSHAPAgreementList
)

TranADSHAPAgreementResults

,Model,Explanation,Index,Case Type,Label,Spearman Correlation
0,TranAD,SHAP,2154,TN,0,0.680157
1,TranAD,SHAP,1647,TN,0,0.742587
2,TranAD,SHAP,161,TN,0,0.581950
3,TranAD,SHAP,1394,TN,0,0.715723
4,TranAD,SHAP,929,TN,0,0.459000
5,TranAD,SHAP,911,TN,0,0.442277
6,TranAD,SHAP,1528,TN,0,0.562164
7,TranAD,SHAP,193,TN,0,0.712404
8,TranAD,SHAP,383,TN,0,0.618466
9,TranAD,SHAP,1878,TN,0,0.600572


In [30]:
TranADSHAPAgreementSummary = (
    TranADSHAPAgreementResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Spearman Correlation"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADSHAPAgreementSummary

,Model,Explanation,Case Type,mean,std,count
0,TranAD,SHAP,TN,0.611530,0.104514,10
1,TranAD,SHAP,TP,0.689374,0.101344,10


In [31]:
# Save the Results

TranADSHAPAgreementResults.to_csv(
    AGREEMENT_DIR / "tranad_shap_reconstruction_error_agreement_results.csv",
    index=False,
)

TranADSHAPAgreementSummary.to_csv(
    AGREEMENT_DIR / "tranad_shap_reconstruction_error_agreement_summary.csv",
    index=False,
)

# Final Reconstruction Error Agreement Evaluation Results

In [32]:
AgreementSummary = pd.concat(
    [
        AnoFormerAttentionAgreementSummary,
        AnoFormerSHAPAgreementSummary,
        TranADAttentionAgreementSummary,
        TranADSHAPAgreementSummary,
    ],
    ignore_index=True,
)

AgreementSummary

,Model,Explanation,Case Type,mean,std,count
0,AnoFormer,Attention,TN,0.280149,0.097898,10
1,AnoFormer,Attention,TP,0.230699,0.091359,10
2,AnoFormer,SHAP,TN,0.316436,0.083348,10
3,AnoFormer,SHAP,TP,0.465382,0.117179,10
4,TranAD,Attention,TN,0.477518,0.139822,10
5,TranAD,Attention,TP,0.583823,0.156952,10
6,TranAD,SHAP,TN,0.611530,0.104514,10
7,TranAD,SHAP,TP,0.689374,0.101344,10


In [33]:
AgreementSummary.to_csv(
    AGREEMENT_DIR / "reconstruction_error_agreement_summary_all_methods.csv",
    index=False,
)